# 04 — Baseline first, then train the existing head
Training uses one positive and up to four unique negatives: nearest wrong, up to two additional same-panel characters, then one farther character. Validation scores **all matched character appearances on the page**, with the same fixed candidates for baseline and every epoch. Texts whose true speaker failed matching are excluded and counted in notebook 03.

This experiment predicts a visible speaker. Unlinked texts supply no labels. There is no narration classifier, decoder training, or Experiment 2.


In [ ]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\ESHAAN\BYTE\Applied-Task\Magi\Manga109Association")
sys.path.insert(0, str(ROOT))
from common import *

DATA.mkdir(exist_ok=True)
CHECKPOINTS.mkdir(exist_ok=True)


In [ ]:
import copy
import random
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 10
BATCH_SIZE = 128
MAX_NEGATIVES = 4
CLOSE_THRESHOLD = 0.10
assert MAX_NEGATIVES >= 1 and EPOCHS >= 1 and BATCH_SIZE >= 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

extraction = read_json(DATA / "extraction_manifest.json")
assert extraction["train_pages_hash"] == digest(read_json(DATA / "train_pages.json"))
assert extraction["val_pages_hash"] == digest(read_json(DATA / "val_pages.json"))


In [ ]:
def training_candidates(page, text_id, rng):
    positive = page["speaker_links"][text_id]
    wrong = [pair_metadata(page, text_id, cid) for cid in sorted(page["characters"]) if cid != positive]
    wrong.sort(key=lambda m: (m["distance"], m["character_id"]))
    selected = []
    if wrong:
        selected.append(wrong[0]["character_id"])
    same_panel = [m["character_id"] for m in wrong if m["same_panel"] and m["character_id"] not in selected]
    selected.extend(same_panel[:2])
    remaining = [m["character_id"] for m in wrong if m["character_id"] not in selected]
    if remaining:
        selected.append(rng.choice(remaining[len(remaining) // 2:]))
    selected = selected[:MAX_NEGATIVES]
    assert len(selected) == len(set(selected)) and positive not in selected
    if wrong:
        assert wrong[0]["character_id"] in selected
    return [positive] + selected


def build_manifest(split):
    rng = random.Random(SEED)
    records = []
    for cache_record in tqdm(extraction["splits"][split], desc=f"{split} candidates"):
        page = load_cached(cache_record)
        texts = []
        for tid in sorted(page["speaker_links"]):
            ids = training_candidates(page, tid, rng) if split == "train" else sorted(page["characters"])
            texts.append({"text_id": tid, "candidate_ids": ids,
                          "pairs": [pair_metadata(page, tid, cid) for cid in ids]})
        if texts:
            records.append({**cache_record, "book": page["book"], "page_index": page["page_index"], "texts": texts})
    assert records, f"No usable {split} pages. Check coverage."
    return {"split": split, "seed": SEED, "max_negatives": MAX_NEGATIVES if split == "train" else None,
            "extraction_hash": digest(extraction), "candidate_hash": digest(records), "pages": records}


train_manifest = build_manifest("train")
val_manifest = build_manifest("val")
assert {p["book"] for p in train_manifest["pages"]}.isdisjoint({p["book"] for p in val_manifest["pages"]})
save_json(DATA / "train_candidates.json", train_manifest)
save_json(DATA / "val_candidates.json", val_manifest)
for manifest in [train_manifest, val_manifest]:
    pairs = [pair for p in manifest["pages"] for t in p["texts"] for pair in t["pairs"]]
    print(manifest["split"], "pages:", len(manifest["pages"]), "pairs:", len(pairs),
          "nearest-wrong negatives:", sum(p["is_nearest_character"] and p["target"] == 0 for p in pairs))


## Load the original head and verify only its parameters are trainable
Use Magi's actual head architecture. After copying the head, release the rest of the model so training uses little GPU memory. Baseline is measured before any optimizer step. The original head checkpoint came directly from frozen local Magi in notebook 03.


In [ ]:
model = load_magi("cpu")
for parameter in model.text_character_matching_head.parameters():
    parameter.requires_grad = True
trainable = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
print("Trainable model parameters:", trainable)
assert trainable and all(name.startswith("text_character_matching_head.") for name in trainable)
head = copy.deepcopy(model.text_character_matching_head).to(DEVICE)
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

original = torch.load(CHECKPOINTS / "original_head.pt", map_location="cpu", weights_only=True)
assert original["extraction_hash"] == digest(extraction)
head.load_state_dict(original["state_dict"], strict=True)

baseline_metrics, baseline_rows, baseline_scores = evaluate(head, val_manifest, DEVICE, CLOSE_THRESHOLD)
print("BASELINE:", baseline_metrics)
save_json(DATA / "baseline.json", {
    "extraction_hash": digest(extraction), "candidate_hash": val_manifest["candidate_hash"],
    "metrics": baseline_metrics, "rows": baseline_rows,
})


In [ ]:
optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = torch.nn.BCEWithLogitsLoss()
history = []
best_accuracy = -1.0

for epoch in range(1, EPOCHS + 1):
    head.train()
    shuffled_pages = list(train_manifest["pages"])
    random.Random(SEED + epoch).shuffle(shuffled_pages)
    loss_sum, pair_count = 0.0, 0
    for record in tqdm(shuffled_pages, desc=f"Epoch {epoch}"):
        page = load_cached(record)
        inputs, targets = [], []
        for item in record["texts"]:
            inputs.append(pair_inputs(page, item["text_id"], item["candidate_ids"]))
            targets.extend(pair["target"] for pair in item["pairs"])
        inputs = torch.cat(inputs)
        targets = torch.tensor(targets, dtype=torch.float32)
        order = torch.randperm(len(targets))
        for start in range(0, len(targets), BATCH_SIZE):
            indices = order[start:start + BATCH_SIZE]
            x, y = inputs[indices].to(DEVICE), targets[indices].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = head(x).flatten()
            loss = criterion(logits, y)
            assert torch.isfinite(loss), "Non-finite training loss."
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            pair_count += len(y)
    metrics, _, _ = evaluate(head, val_manifest, DEVICE, CLOSE_THRESHOLD)
    entry = {"epoch": epoch, "train_loss": loss_sum / pair_count, "val_loss": metrics["loss"],
             "val_top1": metrics["top1"], "val_non_nearest_top1": metrics["non_nearest_top1"]}
    history.append(entry)
    print(entry)
    if metrics["top1"] > best_accuracy:
        best_accuracy = metrics["top1"]
        torch.save({
            "state_dict": {k: v.detach().cpu().clone() for k, v in head.state_dict().items()},
            "epoch": epoch, "metrics": metrics, "extraction_hash": digest(extraction),
            "candidate_hash": val_manifest["candidate_hash"],
            "settings": {"lr": LR, "weight_decay": WEIGHT_DECAY, "epochs": EPOCHS, "seed": SEED, "max_negatives": MAX_NEGATIVES},
        }, CHECKPOINTS / "best_association_head.pt")
    save_json(DATA / "training_history.json", history)

display(pd.DataFrame(history))
plt.plot([r["epoch"] for r in history], [r["train_loss"] for r in history], label="train BCE: sampled negatives")
plt.plot([r["epoch"] for r in history], [r["val_loss"] for r in history], label="validation BCE: all candidates")
plt.xlabel("Epoch")
plt.legend()
plt.show()
print("Best validation Top-1:", best_accuracy)
